# 🔍 Análisis de logs de servidor: ¿cuándo estuvo peor el sistema?
se analiza el archivo `server_logs.csv`
**2️. Objetivo del Reto 🎯:**

    2.1. ¿Cuándo estuvo peor el sistema?
    2.2. ¿Qué servicio fue el más afectado?
    2.3. ¿Qué endpoint fue el más comprometido?
    2.4. ¿Qué tipo de errores dominaron?
    2.5. ¿Qué cambió comparado con el funcionamiento normal (la *línea base*)?

| Concepto | Qué significa |
|---|---|
| **Ventana de 5 minutos** | Agrupamos los eventos en bloques de 5 minutos usando la columna `timestamp_event` |
| **Evento malo** | Un evento con severidad `ERROR` o `CRITICAL`, **o** con `status_code` mayor o igual a 500 |
| **Tasa de eventos malos** (*bad_rate*) | `eventos_malos / eventos_totales` dentro de una ventana |
| **Momento crítico** | La ventana con la tasa más alta, pero solo entre ventanas con **al menos 20 eventos** |
| **Línea base** | Todos los eventos que quedan **fuera** del momento crítico (el comportamiento "normal") |

## 1️ Importar librerías y cargar los datos

In [ ]:
# Pandas: para trabajar con tablas de datos
import pandas as pd
# este es para  hacer graficos
import matplotlib.pyplot as plt
# para revisar si un archivo existe
import os

# Ruta del archivo CSV.
ruta_archivo="datos/server_logs.csv"
if not os.path.exists(ruta_archivo):
    ruta_archivo="server_logs.csv"
    
# en esta parte lee el archivo server_logs.csv y guarda en la variable datos
datos=pd.read_csv(ruta_archivo)
# La columna de fecha viene como texto pasa a fecha real para agrupar por tiempo.
datos=['timestamp_event']= pd.to_datetime(datos['timestamp_event'])
print("✅ Datos cargados correctamente")

## 5 Definiciones Operativas
 Marcar los eventos malos `Bad event`
Creamos una columna nueva llamada `es_evento_malo` que vale:
- `True` si el evento es malo (severidad `ERROR`/`CRITICAL` **o** `status_code >= 500`)
- `False` si no lo es

In [ ]:
# La severidad es error o critical
severidad_grave=datos['severity'].isin(['ERROR','CRITICAL'])
# El codigo de estado es 500 o mayor(error de servidor)
codigo_error=datos['status_code']>=500
# se mesclan ambas condiciones solo se necesita que se cumpla uno 
datos['es_evento_malo']=severidad_grave | codigo_error

total_malos=datos["es_evento_malo"].sum()

print(f"Eventos malos en todo el archivo: {total_malos} de {len(datos)}")
print(f"Porcentaje general de eventos malos: {total_malos / len(datos):.2%}")

## 6.1 Exploración inicial
miramos cómo son los datos: cuántas filas hay, de qué fechas son y si faltan valores.
Las columnas más importantes son:
- `timestamp_event`: cuándo ocurrió el evento
- `service_name`: servicio que generó el log
- `severity`: gravedad (`INFO`, `WARN`, `ERROR`, `CRITICAL`)
- `message`: texto del evento
- `endpoint`: ruta que se consultó
- `status_code`: código de respuesta (200 = bien, 5xx = error del servidor)
- `latency_ms`: tiempo de respuesta en milisegundos

In [ ]:
#Pregunta 1: ¿Cuántos logs hay en total?
print(f"6.1.1 ¿Cuántos logs hay en total?: Total de Logs {len(datos)}")
print(f"---- Filas y Columnas: {datos.shape} ----")
print(f"-> Primer evento: {datos['timestamp_event'].value_counts().head(1)}")
print(f"-> Ultimo evento: {datos['timestamp_event'].value_counts().tail(1)}")
print(f"-> Total de valores vacios en la tabla: {datos.isnull().sum().sum()}")
# muestra las primeras 5 filas para ver como luce la tabla
datos.head()

In [ ]:
# Pregunta 2: ¿Qué gravedad aparece más?
print("6.1.2 ¿Qué gravedad aparece más?:")
print(datos['severity'].value_counts())
conteo_gravedad=(datos['severity'].value_counts())
gravedad_mas_frecuente=conteo_gravedad.idxmax() # idxmax() = nombre del valor más repetido
cantidad_gravedad=conteo_gravedad.max()# max() = cuántas veces aparece
print(f"\nGraveda que Más aparece: {gravedad_mas_frecuente} con {cantidad_gravedad}")


In [ ]:
# Preguntas 3 y 4: servicio con más y con menos registros
print("\n6.1.3 ¿Qué servicio genera más registros?")
print(datos['service_name'].value_counts())
conteo_servicios=datos['service_name'].value_counts()

servicio_con_mas_logs=conteo_servicios.idxmax()
cantidad_mas_logs=conteo_servicios.max()
print(f"el servicio genera más registros es: {servicio_con_mas_logs} con {cantidad_mas_logs} logs")

print("\n6.1.4 ¿Qué servicio genera menos registros?")
servicio_con_menos_logs=conteo_servicios.idxmin()
cantidad_menos_logs=conteo_servicios.min()
print(f"el servicio genera menos registros es: {servicio_con_menos_logs} con {cantidad_menos_logs} logs")

In [ ]:
# Pregunta 5: ¿Cuál es el mensaje más repetido?
print("\n6.1.5 ¿Cuál es el mensaje más repetido?:")
conteo_mensajes=datos['message'].value_counts()
mensaje_mas_repetido=conteo_mensajes.idxmax()
cantidad_mensaje=conteo_mensajes.max()
print(f"El Mensaje más repetido es: '{mensaje_mas_repetido}' con {cantidad_mensaje} veces")

In [ ]:
# 6: ¿Cuál es el mensaje "malo" más repetido?
print("6.1.6 ¿Cuál es el mensaje (MALO) más repetido?")
mensajes_malos=datos.loc[datos['es_evento_malo'],'message']# loc[filtro,columna] toma solo filas que cumplen el filtro
conteo_mensaje_malo=mensajes_malos.value_counts()
mensajes_malos_mas_repetidos=conteo_mensaje_malo.idxmax()
cantidad_mensaje_malo=conteo_mensaje_malo.max()
print(f"Mensaje Malo Más repetido: {mensajes_malos_mas_repetidos} con {cantidad_mensaje_malo} veces")

## 6.2 Calcular la tasa de eventos malos cada 5 minutos

Usamos `pd.Grouper` para agrupar los eventos en ventanas de 5 min. Para cada ventana calculamos:
- `eventos_totales`: cuántos eventos hubo
- `eventos_malos`: cuántos fueron malos
- `latencia_promedio_ms`: tiempo de respuesta promedio
- `tasa_eventos_malos`: eventos malos dividido eventos totales

In [ ]:
#variable de agrupador: un grupo por cada bloque de 5 min
grupo_5min=datos.groupby(pd.Grouper(key='timestamp_event',freq='5min'))

#para cada grupo calculamos las metricas que nos interesan
tabla_ventanas=grupo_5min.agg(
    eventos_totales=('es_evento_malo','size'),#cuenta cuantas filas hay
    eventos_malos=('es_evento_malo','sum'), # suma los True
    latencia_promedio_ms=('latency_ms','mean'), #promedio de latencia
)

# calculamos la tasa dividimos malos entre totales
tabla_ventanas['tasa_eventos_malos']=(
    tabla_ventanas['tasa_eventos_malos']/tabla_ventanas['eventos_totales']
)

print("Cantidad de ventanas de 5 minutos: ", len(tabla_ventanas))
tabla_ventanas.head()

## 6.3 Diagnóstico dentro del momento crítico
Dentro de esa ventana (top 1), genera:

- Tabla: eventos malos por service_name (ranking)
- Tabla: top 5 mensajes en malos eventos
- Tabla: top 5 endpoints más comprometidos

"Elegí 1 criterio y declaralo explícitamente:"

- Por cantidad de eventos malos
- Por cantidad de status_code >= 500
- Por mayor latency_ms promedio

También mostramos el top 5 para ver si hay más de un momento problemático.

In [ ]:
#Paso 1 filtar ventanas con 20 eventos o mas
ventanas_validas=tabla_ventanas[tabla_ventanas['eventos_totales']>=20]
print(f"Ventanas con 20 eventos o mas: {len(ventanas_validas)}")

#Paso 2 ordenar de mayor a menor por tasa y queda el top 5 su tasa mas alto
peores_5= ventanas_validas.sort_values(by='tasa_eventos_malos',ascending=False).head(5)
print("\n Las 5 peores ventanas: ")
print(peores_5[['eventos_totales','eventos_malos','tasa_eventos_malos']])

#paso 3 primero es el momento critico el indice de la tabla es el inicio de ventana
inicio_incidente=peores_5.index[0]
fin_incidente=inicio_incidente+pd.Timedelta(minutes=5)

print(f"\n Momento critico: de {inicio_incidente} a {fin_incidente}")
print(f"\n Tasa de eventos malos {peores_5['tasa_eventos_malos'].iloc[0]:.2%}")

## Diagnóstico dentro del incidente

Ahora nos quedamos solo con los eventos que ocurrieron durante el momento crítico y miramos **solo los eventos malos** para responder:
- ¿Qué servicio sufrió más?
- ¿Qué endpoint fue el más comprometido?
- ¿Qué mensajes y códigos de error dominaron?

In [ ]:
# eventos que ocurrieron durante el incidente (entre inicio y fin de la ventana)
esta_en_incidente=(datos['timestamp_event']>=inicio_incidente)&(datos['timestamp_event']<fin_incidente)
eventos_incidente=(datos[esta_en_incidente])

#De esos, solo le quitamos los malos
errores_incidente=eventos_incidente[eventos_incidente['es_evento_malo']]

print(f"Evento durante el incidente: {len(eventos_incidente)}")
print(f"Evento durante el incidente: {len(errores_incidente)}")

In [ ]:
#contadores por servicio, por end point comprometidos(5) primeros nomas,
print("🔧 Servicio con mas eventos malos: ")
print(errores_incidente['service_name'].value_counts())

print("🌐 End point mas comprometidos: ")
print(errores_incidente['endpoint'].value_counts().head(5))

print("💬 Mensajes de error frecuentes: ")
print(errores_incidente['message'].value_counts().head(5))

print("🔢 Codigo de estado de los eventos Malos: ")
print(errores_incidente['status_code'].value_counts().head())

# 6.4 “Qué cambió” (Incidente vs Baseline)
Comparar momento `crítico vs baseline` con una tabla simple que incluya:
La **línea base**  eventos que quedan **fuera** del incidente. Así sabemos cómo es el sistema "en un día normal" y cuanto cambio con el problema.

Creamos una función `calcular_metricas` para calcular lo mismo en ambos grupos y no repetir código. 
- total_events
- bad_rate
- avg_latency_ms
- %_5xx (porcentaje de status_code >= 500)

In [ ]:
def calcular_metricas(tabla_eventos):
    # recibie una tabla de eventos y devuelve un diccionario con sus metricas
    total=len(tabla_eventos)
    malos=tabla_eventos['es_evento_malo'].sum()

    return{
    'eventos_totales': total,
    'eventos_malos': malos,
    'tasa_eventos_malos': malos/total,
    'latencia_promedio_ms': tabla_eventos['latency_ms'].mean,
    # % de eventos con codigo 5xx (500 o mas)
    'porcentaje_5xx':(tabla_eventos['status_code']>=500).mean()*100
    }

# linea base = todo lo que no esta en el incidente
# (el simbolo ~ significa lo contrario de)

evento_linea_base = datos[~esta_en_incidente]

metrica_incidente = calcular_metricas(eventos_incidente)
metrica_linea_base = calcular_metricas(evento_linea_base)

# Juntamos ambas metricas en una sola tabla para comparar
comparacion= pd.DataFrame({
    'Incidente': metrica_incidente,
    'Linea base': metrica_linea_base
})

#agregamos una columna que dice cuantas veces es mayor el incidente
#Solo tiene sentido para la tasa, la latencia y el % de 5xx
#comparar "cantidad de eventos" 
fila_comparables= ['tasa_eventos_malos','latencia_promedio_ms','porcentaje_5xx']
comparacion['Veces mayor']= comparacion.loc[fila_comparables,'Incidente']/comparacion.loc[fila_comparables,'Linea base']

comparacion.round(2)

# Para entender el contraste: ¿qué errores eran "normales" en la línea base?

In [ ]:
errores_lineas_base=evento_linea_base[evento_linea_base['es_evento_malo']]

print("Servicios con más errores en la linea base: ")
print(errores_lineas_base['service_name'].value_counts().head(3))

print("\n Mensaje de error mas comunnes de la linea base: ")